# HateMirage @ ICON 2026 - 2xT4 QLoRA (dual-GPU, HF-sync, resume, auto-eval)

**Tasks:** A=Target span - B=Intent+Implication generation - **Metrics:** SBERT-cosine + ROUGE-L F1 (greedy T=0)  
**Setup:** Kaggle accelerator `GPU T4 x2` - Internet ON - Secrets: `HF_TOKEN` (write) - Optional: `WANDB_API_KEY`  
**Flow:** env-check -> install -> HF login+config -> data -> RAG -> torchrun-DDP train (both T4s) -> Hub push -> inference -> eval -> Codabench zips.  
If the session hits the 9h limit or converges early, training saves + pushes and the notebook jumps straight to eval - never lose a run. Resume anytime from `HF_REPO`.

Paper: https://arxiv.org/abs/2603.02684 - Task: https://sai-kartheek-reddy.github.io/HateMirage-ICON2026/ - Codabench: https://www.codabench.org/competitions/17783/

In [ ]:
# CELL 1 - env check: must see 2x Tesla T4
import torch, os, time, shutil
T0 = time.time()
print('torch', torch.__version__, '| cuda?', torch.cuda.is_available())
NGPU = torch.cuda.device_count()
print('GPUs:', NGPU)
for i in range(NGPU):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")
!nvidia-smi -L
!df -h /kaggle/working | tail -1
assert NGPU >= 1, 'Enable GPU T4 x2 under Settings > Accelerator'
DUAL = NGPU >= 2
print('DUAL-GPU mode:', DUAL)
OUT = '/kaggle/working/hatemirage_out'
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT
print('OUT:', OUT)

In [ ]:
# CELL 2 - deps (Kaggle default torch 2.x; keep bnb compatible)
!pip install -q -U "transformers>=4.44" accelerate peft trl bitsandbytes "datasets>=2.20" sentence-transformers rouge-score faiss-cpu huggingface_hub pandas scikit-learn openpyxl 2>&1 | tail -2
import transformers, peft, trl, bitsandbytes
print('transformers', transformers.__version__)

In [ ]:
# CELL 3 - secrets (resilient: smoke mode if HF_TOKEN absent) + run config
import os
HF_TOKEN_HARDCODED = "hf_YOUR_TOKEN_HERE"  # private fallback; Secrets win
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN secret: present')
except Exception as e:
    HF_TOKEN = HF_TOKEN_HARDCODED
    print('HF_TOKEN secret: ABSENT -> using built-in fallback.', str(e)[:100])
    print('HF_TOKEN: unexpected empty (cleared fallback).', str(e)[:100])
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
HAS_HF = bool(HF_TOKEN)
if HAS_HF:
    get_ipython().system('huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential 2>&1 | tail -1')

# ---- EDIT: your Hub namespace + model choice ----
HF_USER = 'blackpanter_252'
if HAS_HF:
    try:
        from huggingface_hub import whoami as _who3
        HF_USER = _who3(token=HF_TOKEN).get('name', HF_USER)
        print('HF user:', HF_USER)
    except Exception as _e3:
        print('whoami failed, keep HF_USER:', HF_USER, str(_e3)[:100])
MODEL_ID = 'microsoft/Phi-3-mini-128k-instruct'
SMOKE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
EFFECTIVE_MODEL = MODEL_ID if HAS_HF else SMOKE_MODEL
HF_REPO  = f'{HF_USER}/hatemirage-phi3-qlora'
WITH_RAG, MAX_RUNTIME_H = True, 8.2
EPOCHS, BS, GRAD_ACCUM, LORA_R = (3, 2, 8, 64) if HAS_HF else (1, 2, 2, 8)
print('model:', EFFECTIVE_MODEL, '| hub:', HF_REPO if HAS_HF else '(skipped, no token)', '| RAG:', WITH_RAG)
if HAS_HF:
    from huggingface_hub import whoami
    print(whoami(token=HF_TOKEN)['name'])


In [ ]:
# CELL 4 - official task data (Development + Evaluation phases, pinned SHAs)
# Train.xlsx 04cb07ab | Val-for-dev a81fe96d | Val-Data-with-labels 838c3f83 | test-data-for-eval d76f741f
import glob, pandas as pd, os, urllib.request
GH = 'https://raw.githubusercontent.com/Sai-Kartheek-Reddy/HateMirage-ICON2026/main/'
OFFICIAL = {
 'train': 'Development%20Phase/Train.xlsx',
 'devval': 'Development%20Phase/Val-for-dev.xlsx',
 'vallab': 'Evaluation%20Phase/Val-Data-with-labels.xlsx',
 'test': 'Evaluation%20Phase/test-data-for-eval.xlsx',
}
os.makedirs(f'{OUT}/official', exist_ok=True)
def fetch_official(key):
    p = f'{OUT}/official/{OFFICIAL[key].split("/")[-1]}'
    if not os.path.exists(p):
        urllib.request.urlretrieve(GH + OFFICIAL[key], p)
        print('downloaded', p)
    return p
def load_off(p, labeled):
    df = pd.read_excel(p, engine='openpyxl')
    df.columns = [c.strip() for c in df.columns]
    out = pd.DataFrame({'Index': df['Index'], 'comment': df['Comments'].astype(str)})
    if labeled:
        out['target'] = df['Target'].astype(str)
        out['intent'] = df['Intent'].astype(str)
        out['implication'] = df['Implication'].astype(str)
    return out.dropna().reset_index(drop=True)
try:
    train_df = load_off(fetch_official('train'), True)
    val_df = load_off(fetch_official('vallab'), True)
    devval_df = load_off(fetch_official('devval'), False)
    test_df = load_off(fetch_official('test'), False)
    print('OFFICIAL data:', len(train_df), len(val_df), len(devval_df), len(test_df))
except Exception as e:
    print('official fetch failed, fallback chain:', str(e)[:200])
    cands = glob.glob('/kaggle/input/**/*.*', recursive=True)[:10]
    print('kaggle inputs:', cands)
    smoke = {'Index': range(12), 'comment': ['The virus came from [Country] labs; they engineered it to destroy us.']*12,
             'target': ['[Country]']*12, 'intent': ['To accuse [Country] of building a bioweapon.']*12,
             'implication': ['Could incite hostility against [Country] citizens.']*12}
    train_df = pd.DataFrame(smoke); val_df = train_df.iloc[:4].copy()
    devval_df = train_df[['Index','comment']].iloc[:4].copy()
    test_df = train_df[['Index','comment']].iloc[:4].copy()
for n, df in [('train',train_df),('val',val_df),('devval',devval_df),('test',test_df)]:
    print(n, df.shape, list(df.columns))
    display(df.head(2))
train_df.to_csv(f'{OUT}/train_norm.csv', index=False); val_df.to_csv(f'{OUT}/val_norm.csv', index=False)
devval_df.to_csv(f'{OUT}/devval_norm.csv', index=False); test_df.to_csv(f'{OUT}/test_norm.csv', index=False)
print('normalized ->', OUT)


In [ ]:
# CELL 5 - RAG index (FAISS over fact-check refs; falls back to train intents)
import glob, json, urllib.request, numpy as np
try:
    _rj = f'{OUT}/official/RAG_Reference_Data.jsonl'
    urllib.request.urlretrieve(GH + 'Starter-Kit/source_docs/RAG_Reference_Data.jsonl', _rj)
    print('downloaded official RAG refs')
except Exception as _e5:
    print('rag-dl note:', str(_e5)[:120])
ref = glob.glob(f'{OUT}/official/RAG_Reference_Data.jsonl')
ref += glob.glob('/kaggle/input/**/RAG_Reference_Data.jsonl', recursive=True)
ref += glob.glob('source_docs/RAG_Reference_Data.jsonl')
print('ref jsonl:', ref)
passages = []
if ref:
    for line in open(ref[0]):
        try: passages.append(json.loads(line).get('text', '').strip() or line.strip())
        except: passages.append(line.strip())
if len(passages) < 50:
    passages += train_df['comment'].tolist()[:2000]  # fallback corpus
print('passages:', len(passages))
from sentence_transformers import SentenceTransformer
import faiss
emb_model = 'sentence-transformers/all-mpnet-base-v2'
enc = SentenceTransformer(emb_model)
E = enc.encode(passages, show_progress_bar=True, normalize_embeddings=True).astype('float32')
idx = faiss.IndexFlatIP(E.shape[1]); idx.add(E)
faiss.write_index(idx, f'{OUT}/rag_index.faiss')
json.dump({'passages': passages[:5000]}, open(f'{OUT}/rag_passages.json','w'))
print('FAISS saved', idx.ntotal)

def retrieve(q, k=5):
    qv = enc.encode([q], normalize_embeddings=True).astype('float32')
    _, I = idx.search(qv, k)
    return ' | '.join(passages[i][:300] for i in I[0])
train_df['evidence'] = train_df['comment'].apply(retrieve) if WITH_RAG else ''
val_df['evidence'] = val_df['comment'].apply(retrieve) if WITH_RAG else ''
test_df['evidence'] = test_df['comment'].apply(retrieve) if WITH_RAG else ''
devval_df['evidence'] = devval_df['comment'].apply(retrieve) if WITH_RAG else ''
test_df.to_csv(f'{OUT}/test_norm.csv', index=False)
devval_df.to_csv(f'{OUT}/devval_norm.csv', index=False)
train_df.to_csv(f'{OUT}/train_norm.csv', index=False); val_df.to_csv(f'{OUT}/val_norm.csv', index=False)
print('evidence attached; e.g.:', str(val_df['evidence'].iloc[0])[:200])

In [ ]:
# CELL 6a - training config (DDP on BOTH T4s via torchrun in next cell)
import os, json, glob
os.makedirs('src', exist_ok=True)
print('src/train_ddp.py present:', os.path.exists('src/train_ddp.py'))
cfg = {'model_id': EFFECTIVE_MODEL, 'hub_id': (HF_REPO if HAS_HF else None),
       'train_csv': f'{OUT}/train_norm.csv',
       'val_csv': f'{OUT}/val_norm.csv', 'output_dir': f'{OUT}/ckpt',
       'with_rag': WITH_RAG, 'epochs': EPOCHS, 'bs': BS,
       'grad_accum': GRAD_ACCUM, 'lora_r': LORA_R,
       'eval_steps': 200, 'save_steps': 200, 'max_seq': 1536,
       'max_runtime_h': MAX_RUNTIME_H, 'resume_ckpt': None, 'warm_adapter': None}
# resume from Hub if a prior session pushed adapters
if HAS_HF:
    try:
        from huggingface_hub import snapshot_download
        ck = snapshot_download(HF_REPO, token=os.environ['HF_TOKEN'])
        import re as _re, json as _js
        states = glob.glob(ck + '/**/trainer_state.json', recursive=True)
        def _step(p):
            m = _re.search(r'checkpoint-(\d+)', p)
            return int(m.group(1)) if m else -1
        states = sorted(states, key=_step)
        cfg['resume_ckpt'] = None
        want = {'model_id': EFFECTIVE_MODEL, 'lora_r': LORA_R,
                'n_train': len(train_df) * 3, 'epochs': EPOCHS,
                'eff_batch': BS * (2 if globals().get('DUAL', True) else 1) * GRAD_ACCUM}
        try:
            info = None
            for cand_info in [os.path.join(ck, 'RESUME_INFO.json'),
                              os.path.join(ck, 'ckpt', 'RESUME_INFO.json')]:
                if os.path.exists(cand_info):
                    info = _js.load(open(cand_info))
                    break
            ok = bool(states) and info is not None and all(info.get(k) == v for k, v in want.items())
            print(f"resume fingerprint want={want} got={info} compat={ok}")
            if ok:
                cfg['resume_ckpt'] = os.path.dirname(states[-1])
                cfg['warm_adapter'] = os.path.dirname(states[-1])
        except Exception as _eac:
            print('resume check failed -> fresh:', str(_eac)[:150])
        print('resume_ckpt:', cfg['resume_ckpt'])

    except Exception as e:
        cfg['resume_ckpt'] = None
        print('no prior Hub ckpt (fresh start):', str(e)[:200])
else:
    print('resume skipped (no HF_TOKEN)')
json.dump(cfg, open(f'{OUT}/train_config.json', 'w'), indent=2)
print(json.dumps(cfg, indent=2))


In [ ]:
%%writefile src/train_ddp.py
"""train_ddp.py - QLoRA SFT for HateMirage, launched via torchrun (DDP, 2xT4).
Launched by the Kaggle notebook as:
  torchrun --nproc_per_node=2 --master_port=29511 src/train_ddp.py --config out/train_config.json
Single-GPU fallback: python src/train_ddp.py --config ... (works, slower).
Env: HF_TOKEN must be set (Kaggle Secrets). Expects train/val CSVs with
  id,comment,target,intent,implication (+ optional evidence).
Pushes adapters to Hub every `push_every` steps if --hub_id given.
Timeout guard: --max_runtime_h (default 8.2) -> saves + exits 42 so notebook jumps to eval.
"""
import argparse, json, os, sys, time, math
import pandas as pd
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, EarlyStoppingCallback)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

PROMPT_VERSION = "v2-official-format"  # canonical text: prompts/parts.json (organizers' Zero-Shot.py/RAG.py format)
SYS_INTRO = "You are an expert in analyzing hateful comments driven by fake narratives.\n\nBased on the given comment, provide the requested analysis."
RAG_INTRO_TPL = "You are an expert in analyzing hateful comments driven by fake narratives.\n\nThe following context provides background information retrieved from external sources:\n[Context]: \"{context}\"\n\nBased on the context and the comment, provide the requested analysis."
TASK_TARGET = "If there is one target, only mention that. If there are multiple, mention each target as a single word and separate them by commas."
TASK_INTENT = "Briefly describe the *Intent* in a single concise sentence."
TASK_IMPLICATION = "Briefly describe the possible *Implication* in a single concise sentence."

def _intro(ev):
    return RAG_INTRO_TPL.replace("{context}", ev[:1200]) if ev else SYS_INTRO

def _norm_intent(s):
    s = str(s or "").strip()
    if s[:3].lower() == "to ":
        s = s[3:].lstrip()
    return "To " + s

def to_prompt(row, with_rag: bool):
    ev_raw = str(row.get("evidence", "")).strip() if with_rag else ""
    out = {}
    for task, label in (("TARGET", "Target"), ("INTENT", "Intent"), ("IMPLICATION", "Implication")):
        task_txt = {"TARGET": TASK_TARGET, "INTENT": TASK_INTENT, "IMPLICATION": TASK_IMPLICATION}[task]
        prompt = f"{_intro(ev_raw)}\n## Comment: \"{row['comment']}\"\n## Task: {task_txt}\n## {label}:"
        out[task.lower()] = prompt + " "
    out["target"] += f"{str(row['target']).strip()} </s>"
    out["intent"] += f"{_norm_intent(row['intent'])} </s>"
    out["implication"] += f"{str(row['implication']).strip()} </s>"
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    a = ap.parse_args()
    cfg = json.load(open(a.config))
    t0 = time.time()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)  # each DDP rank loads its replica on its own GPU
    is_main = int(os.environ.get("RANK", "0")) == 0
    if is_main: print(f"[train_ddp] config={cfg} world={os.environ.get('WORLD_SIZE','1')}", flush=True)

    train_df = pd.read_csv(cfg["train_csv"])
    val_df = pd.read_csv(cfg["val_csv"])
    with_rag = cfg.get("with_rag", True)
    rows = []
    for _, r in train_df.iterrows():
        for k, v in to_prompt(r, with_rag).items():
            rows.append({"text": v})
    train_ds = Dataset.from_list(rows)
    vrows = []
    for _, r in val_df.iterrows():
        for k, v in to_prompt(r, with_rag).items():
            vrows.append({"text": v})
    val_ds = Dataset.from_list(vrows)

    tok = AutoTokenizer.from_pretrained(cfg["model_id"], token=(os.environ.get("HF_TOKEN") or None), trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)
    _load_kw = dict(quantization_config=bnb,
        token=(os.environ.get("HF_TOKEN") or None), trust_remote_code=True,
        attn_implementation="eager")
    from transformers import AutoConfig as _AC
    _mcfg = _AC.from_pretrained(cfg["model_id"], token=(os.environ.get("HF_TOKEN") or None),
                                trust_remote_code=True)
    _mcfg.torch_dtype = torch.float16  # checkpoint may default to bf16 (T4-hostile)
    try:
        model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], config=_mcfg,
                                                     dtype=torch.float16, **_load_kw)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], config=_mcfg,
                                                     torch_dtype=torch.float16, **_load_kw)
        if is_main: print("[train_ddp] from_pretrained fallback (torch_dtype)", flush=True)
    try:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    except TypeError:
        model = prepare_model_for_kbit_training(model)
        if is_main: print("[train_ddp] kbit_training fallback (no gradient_checkpointing kw)", flush=True)
    model.config.use_cache = False
    _n_bf16 = 0
    for _n, _p in model.named_parameters():
        if _p.requires_grad and _p.dtype == torch.bfloat16:
            _p.data = _p.data.to(torch.float32); _n_bf16 += 1
    if is_main: print(f"[train_ddp] cast {_n_bf16} trainable bf16 params to fp32", flush=True)
    peft = LoraConfig(r=cfg.get("lora_r", 64), lora_alpha=cfg.get("lora_alpha", 16), lora_dropout=0.05,
                      bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    # Warm-start: load adapter WEIGHTS only (fresh optimizer). Optimizer-state resume
    # crashes bitsandbytes kernels, so resume_ckpt (full state) is intentionally unsupported.
    from peft import PeftModel as _PM
    _warm_dir = cfg.get("warm_adapter") or None
    _warm_ok = False
    if _warm_dir:
        import glob as _gg
        _ad = _gg.glob(os.path.join(_warm_dir, "adapter_model.safetensors"))
        if _ad:
            try:
                model = _PM.from_pretrained(model, _warm_dir)
                _warm_ok = True
                if is_main: print(f"[train_ddp] warm-started adapter weights from {_warm_dir}", flush=True)
            except Exception as e:
                print(f"[train_ddp] warm load failed ({e}), random init", flush=True)
        else:
            print(f"[train_ddp] warm dir lacks adapter: {_warm_dir}", flush=True)
    import inspect as _insp
    _ta_params = set(_insp.signature(TrainingArguments.__init__).parameters)
    _strat_key = "eval_strategy" if "eval_strategy" in _ta_params else "evaluation_strategy"
    _world = max(1, int(os.environ.get("WORLD_SIZE", "1")))
    _est_steps = max(1, (len(train_ds) * cfg.get("epochs", 3)) // max(1, (cfg.get("bs", 2) * _world * cfg.get("grad_accum", 8))))
    _ev = int(min(cfg.get("eval_steps", 200), max(5, _est_steps)))
    _do_best = _ev < _est_steps  # need >=1 eval before end for best-model + early-stop
    if is_main: print(f"[train_ddp] est_steps={_est_steps} eval_every={_ev} load_best={_do_best}", flush=True)
    _fingerprint = {"model_id": cfg["model_id"], "lora_r": cfg.get("lora_r", 64),
                    "n_train": len(train_ds), "epochs": cfg.get("epochs", 3),
                    "eff_batch": cfg.get("bs", 2) * _world * cfg.get("grad_accum", 8),
                    "max_steps": _est_steps}
    os.makedirs(cfg["output_dir"], exist_ok=True)
    json.dump(_fingerprint, open(os.path.join(cfg["output_dir"], "RESUME_INFO.json"), "w"), indent=1)
    _ta_want = dict(
        output_dir=cfg["output_dir"], num_train_epochs=cfg.get("epochs", 3),
        per_device_train_batch_size=cfg.get("bs", 2), per_device_eval_batch_size=2,
        gradient_accumulation_steps=cfg.get("grad_accum", 8),
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
        optim="paged_adamw_32bit", fp16=False, bf16=False,
        gradient_checkpointing=True, logging_steps=25, eval_steps=_ev,
        save_steps=_ev, **{_strat_key: "steps"}, save_strategy="steps",
        save_total_limit=3, load_best_model_at_end=_do_best, metric_for_best_model="eval_loss",
        greater_is_better=False, ddp_find_unused_parameters=False,
        # NOTE: Hub push is notebook-side (manual, guarded). Trainer-side push is OFF:
        # SFTTrainer.__init__ calls create_repo and 403-crashes when the token lacks
        # write rights on the namespace. Training must never depend on Hub writes.
        push_to_hub=False, report_to="none", seed=42,
        max_grad_norm=0.3, weight_decay=0.01,
    )
    _dropped = sorted(k for k in _ta_want if k not in _ta_params)
    _ta_want = {k: v for k, v in _ta_want.items() if k in _ta_params}
    if is_main: print(f"[train_ddp] dropped TrainingArguments keys: {_dropped}", flush=True)
    args = TrainingArguments(**_ta_want)
    # timeout callback (holder: TrainerState has no save_model; use the trainer)
    from transformers import TrainerCallback
    _holder = {}
    class TimeoutCB(TrainerCallback):
        def on_step_end(self, args, state, control, **kwargs):
            hrs = (time.time() - t0) / 3600
            if hrs > cfg.get("max_runtime_h", 8.2):
                print(f"[train_ddp] TIMEOUT {hrs:.2f}h -> save+exit42", flush=True)
                tr = _holder.get("trainer")
                if tr is not None:
                    try:
                        tr.save_model(cfg["output_dir"])
                    except Exception as e:
                        print("[train_ddp] timeout save failed:", e, flush=True)
                sys.stdout.flush(); os._exit(42)
    _sft_params = set(_insp.signature(SFTTrainer.__init__).parameters)
    _sft_kw = dict(args=args)
    if not _warm_ok:
        _sft_kw["peft_config"] = peft
    if "dataset_text_field" in _sft_params:
        _sft_kw["dataset_text_field"] = "text"
    if "tokenizer" in _sft_params:
        _sft_kw["tokenizer"] = tok
    elif "processing_class" in _sft_params:
        _sft_kw["processing_class"] = tok
    if "max_seq_length" in _sft_params:
        _sft_kw["max_seq_length"] = cfg.get("max_seq", 1536)
    elif "max_length" in _sft_params:
        _sft_kw["max_length"] = cfg.get("max_seq", 1536)
    if "packing" in _sft_params:
        _sft_kw["packing"] = False
    _cbs = [TimeoutCB()]
    if _do_best:
        _cbs.append(EarlyStoppingCallback(early_stopping_patience=2))
    _sft_kw["callbacks"] = _cbs
    os.makedirs(cfg["output_dir"], exist_ok=True)
    try:
        trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds, **_sft_kw)
        _holder["trainer"] = trainer
    except Exception as e:
        import traceback as _tb
        open(os.path.join(cfg["output_dir"], "TRAIN_ERROR.txt"), "w").write(
            f"SFTTrainer build failed: {e}\nkwargs={sorted(_sft_kw)}\n" + _tb.format_exc())
        print("[train_ddp] SFTTrainer build failed, wrote TRAIN_ERROR.txt", flush=True)
        raise
    _n_cast = 0
    for _n, _p in trainer.model.named_parameters():
        if _p.requires_grad and _p.dtype != torch.float32:
            _p.data = _p.data.to(torch.float32); _n_cast += 1
    if is_main: print(f"[train_ddp] post-peft cast {_n_cast} trainable params to fp32", flush=True)
    _resume = None  # full-state resume disabled (bnb optimizer-state crash); warm-start above instead
    if cfg.get("resume_ckpt"):
        print(f"[train_ddp] ignoring resume_ckpt (warm-start used): {cfg.get('resume_ckpt')}", flush=True)
    _fatal = None
    _fatal_tb = ""
    try:
        trainer.train(resume_from_checkpoint=_resume)
    except SystemExit:
        raise
    except Exception as e:
        import traceback as _tbr
        if _resume:
            print(f"[train_ddp] train-with-resume failed ({type(e).__name__}: {e}), retrying FRESH", flush=True)
            try:
                open(os.path.join(cfg["output_dir"], "TRAIN_ERROR.txt"), "a").write(
                    f"\nresume attempt failed, fresh retry: {e}\n" + _tbr.format_exc(limit=6))
            except Exception:
                pass
            _resume = None
            try:
                trainer.train(resume_from_checkpoint=None)
            except SystemExit:
                raise
            except Exception as e2:
                _fatal, _fatal_tb = e2, _tbr.format_exc()
        else:
            _fatal, _fatal_tb = e, _tbr.format_exc()
    if _fatal is not None:
        try:
            open(os.path.join(cfg["output_dir"], "TRAIN_ERROR.txt"), "a").write(
                f"\ntrainer.train failed: {_fatal}\n{_fatal_tb}")
        except Exception:
            pass
        print("[train_ddp] ERROR, saving partial:", _fatal, flush=True)
        trainer.save_model(cfg["output_dir"] + "-partial")
        raise _fatal
    trainer.save_model(cfg["output_dir"])
    if is_main:
        tok.save_pretrained(cfg["output_dir"])
        hrs = (time.time() - t0) / 3600
        json.dump({"hours": hrs, "hub": cfg.get("hub_id")}, open(os.path.join(cfg["output_dir"], "train_done.json"), "w"), indent=2)
        print(f"[train_ddp] DONE {hrs:.2f}h", flush=True)

if __name__ == "__main__":
    main()


In [ ]:
%%bash
echo "OUT=$OUT"
ls -la src/train_ddp.py
if [ -f src/train_ddp.py ]; then
  N=$(nvidia-smi -L | wc -l)
  echo "GPUs for torchrun: $N"
  if [ "$N" -ge 2 ]; then
    torchrun --nproc_per_node=2 --master_port=29511 src/train_ddp.py --config $OUT/train_config.json 2>&1 | tee $OUT/ddp.log; echo "torchrun_exit=${PIPESTATUS[0]}" | tee -a $OUT/ddp.log
  else
    python src/train_ddp.py --config $OUT/train_config.json 2>&1 | tee $OUT/ddp.log; echo "torchrun_exit=${PIPESTATUS[0]}" | tee -a $OUT/ddp.log
  fi
else echo "MISSING src/train_ddp.py - writefile cell did not run"; fi
ls -la $OUT/ > $OUT/ckpt_listing.txt 2>&1; ls -la $OUT/ckpt >> $OUT/ckpt_listing.txt 2>&1
tail -5 $OUT/ddp.log 2>/dev/null; cat $OUT/ckpt_listing.txt 2>/dev/null | head -20


In [ ]:
# CELL 7 - push adapter (if HF_TOKEN) + load for inference (balanced shard, both GPUs)
import os, glob, json
def _st(s):
    try:
        open(f'{OUT}/stages.txt', 'a').write(s + '\n')
    except Exception:
        pass
_st('c7-start')
from huggingface_hub import HfApi
api = HfApi()
if HAS_HF:
    try:
        from huggingface_hub import whoami as _who
        _me = _who(token=os.environ['HF_TOKEN']).get('name', '?')
    except Exception as _e:
        _me = 'whoami-failed'
    json.dump({'hf_user': _me, 'hub_repo': HF_REPO},
              open(f'{OUT}/hub_diag.json', 'w'))
    print('hub_diag user:', _me, 'repo:', HF_REPO)
    try:
        import glob as _g7
        _has_ada = bool(_g7.glob(f"{OUT}/ckpt/**/adapter_model.safetensors", recursive=True))
        print('local adapter present:', _has_ada)
        assert _has_ada, 'skip hub push (no adapter trained this run)'
        api.create_repo(HF_REPO, exist_ok=True, token=os.environ['HF_TOKEN'])
        _ci = api.upload_folder(folder_path=f"{OUT}/ckpt", repo_id=HF_REPO,
                                token=os.environ['HF_TOKEN'],
                                ignore_patterns=["optimizer.pt", "rng_state_*.pth"])
        open(f'{OUT}/stages.txt', 'a').write('hub-push-oid: ' + str(getattr(_ci, 'oid', _ci))[:20] + '\n')
        print('pushed ckpt ->', HF_REPO)
    except Exception as e:
        print('push note:', str(e)[:300])
else:
    print('hub push skipped (no HF_TOKEN)')
print(glob.glob(f'{OUT}/ckpt/**/*', recursive=True)[:10])
_st('c7-push-done')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
tok = AutoTokenizer.from_pretrained(EFFECTIVE_MODEL, token=os.environ['HF_TOKEN'] or None,
                                    trust_remote_code=False)
_st('c7-tok')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
from transformers import BitsAndBytesConfig as _BBC, AutoConfig as _AC
_acfg = _AC.from_pretrained(EFFECTIVE_MODEL, token=os.environ['HF_TOKEN'] or None,
                           trust_remote_code=False)
_acfg.torch_dtype = torch.float16
_bbc = _BBC(load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
import gc as _gc, subprocess as _sp
_gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass
try:
    _smi = _sp.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv'],
                   capture_output=True, text=True, timeout=30).stdout.strip()
    open(f'{OUT}/stages.txt', 'a').write('mem-preload: ' + _smi.replace(chr(10), '|') + '\n')
except Exception as _e_smi:
    open(f'{OUT}/stages.txt', 'a').write('mem-preload-fail\n')
import traceback as _tb7
try:
    base = AutoModelForCausalLM.from_pretrained(
        EFFECTIVE_MODEL, config=_acfg, quantization_config=_bbc, dtype=torch.float16,
        device_map='balanced', token=os.environ['HF_TOKEN'] or None,
        trust_remote_code=False, attn_implementation='sdpa')
    _st('c7-base-primary')
except Exception as _e1:
    open(f'{OUT}/stages.txt', 'a').write('base-primary-fail: ' + str(_e1)[:300] + '\n')
    try:
        base = AutoModelForCausalLM.from_pretrained(
            EFFECTIVE_MODEL, quantization_config=_bbc, device_map='auto',
            token=os.environ['HF_TOKEN'] or None, trust_remote_code=False)
        _st('c7-base-fallback')
    except Exception as _e2:
        open(f'{OUT}/c7_error.txt', 'w').write(
            'primary: ' + str(_e1)[:800] + '\nfallback: ' + str(_e2)[:800] + '\n'
            + _tb7.format_exc(limit=12))
        raise
model = base
_st('c7-base')
import re as _re7
def _ckstep(p):
    m = _re7.search(r'checkpoint-(\d+)', p)
    return int(m.group(1)) if m else -1
_local_ckpts = sorted(glob.glob(f'{OUT}/ckpt/checkpoint-*'), key=_ckstep)
_final_ck = f'{OUT}/ckpt' if glob.glob(f'{OUT}/ckpt/adapter_model.safetensors') else None
_cands = _local_ckpts + ([_final_ck] if _final_ck else [])
_loaded = False
for _cd in reversed(_cands):
    try:
        model = PeftModel.from_pretrained(base, _cd)
        print('adapter loaded from LOCAL:', _cd.split('/')[-1])
        _st('c7-adapter-local:' + _cd.split('/')[-1])
        _loaded = True
        break
    except Exception as _e_loc:
        print('local adapter failed:', _cd.split('/')[-1], str(_e_loc)[:150])
if not _loaded and HAS_HF:
    try:
        model = PeftModel.from_pretrained(base, HF_REPO, token=os.environ['HF_TOKEN'])
        print('adapter loaded from Hub')
        _st('c7-adapter-hub')
        _loaded = True
    except Exception as e:
        print('Hub adapter missing:', str(e)[:200])
if not _loaded:
    model = base
    _st('c7-adapter-base')
model.eval()
_st('c7-done')
print('inference ready on', torch.cuda.device_count(), 'GPU(s)')


In [ ]:
# CELL 8 - inference (FINAL prompt v2-official-format, greedy T=0, OOM-safe)
import torch, json, re, pandas as pd
SYS_INTRO = "You are an expert in analyzing hateful comments driven by fake narratives.\n\nBased on the given comment, provide the requested analysis."
RAG_INTRO_TPL = "You are an expert in analyzing hateful comments driven by fake narratives.\n\nThe following context provides background information retrieved from external sources:\n[Context]: \"{context}\"\n\nBased on the context and the comment, provide the requested analysis."
TASK = {
    'TARGET': "If there is one target, only mention that. If there are multiple, mention each target as a single word and separate them by commas.",
    'INTENT': "Briefly describe the *Intent* in a single concise sentence.",
    'IMPLICATION': "Briefly describe the possible *Implication* in a single concise sentence.",
}
def _intro(ev):
    ev = str(ev or '').strip()
    return RAG_INTRO_TPL.replace('{context}', ev[:1200]) if (WITH_RAG and ev) else SYS_INTRO
def prompt_for(comment, ev, task):
    label = {'TARGET': 'Target', 'INTENT': 'Intent', 'IMPLICATION': 'Implication'}[task]
    return f"{_intro(ev)}\n## Comment: \"{comment}\"\n## Task: {TASK[task]}\n## {label}:"
LIMITS = {'TARGET': 96, 'INTENT': 96, 'IMPLICATION': 96}
@torch.inference_mode()
def gen(prompt, max_new):
    try:
        _dev = next(model.parameters()).device
    except Exception:
        import torch as _t
        _dev = _t.device('cuda:0' if _t.cuda.is_available() else 'cpu')
    inp = tok(prompt, return_tensors='pt', truncation=True, max_length=1400)
    inp = {k: v.to(_dev) for k, v in inp.items()}
    try:
        try:
            out = model.generate(**inp, max_new_tokens=max_new, do_sample=False, temperature=0,
                                 pad_token_id=tok.eos_token_id, repetition_penalty=1.0)
        except Exception as _eg:
            if 'seen_tokens' in str(_eg) or 'Cache' in str(type(_eg)) or 'cache' in str(_eg).lower():
                out = model.generate(**inp, max_new_tokens=max_new, do_sample=False, temperature=0,
                                     pad_token_id=tok.eos_token_id, repetition_penalty=1.0,
                                     use_cache=False)
            else:
                raise
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return 'UNRELATED_FALLBACK'
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
def _sents(t, n=2):
    ss = re.findall(r'[^.?!]*[.?!]', t)
    return ' '.join(ss[:n]).strip() if ss else t.splitlines()[0].strip() if t.strip() else t
def clean(field, t):
    t = re.sub(r'\s+', ' ', t).strip().strip('"')
    if field == 'TARGET': t = t.split('\n')[0][:80]
    if field in ('INTENT', 'IMPLICATION'): t = _sents(t, 2)
    if field == 'INTENT' and not t.lower().startswith('to'): t = 'To ' + t
    if field == 'IMPLICATION' and not re.match(r'^(could|may|risks|might)', t.lower()): t = 'Could ' + t[0].lower() + t[1:]
    return t if t else ('Unknown' if field=='TARGET' else ('To spread a misleading narrative.' if field=='INTENT' else 'Could erode trust.'))

open(f'{OUT}/stages.txt', 'a').write('c8-start\n')
def run(df, split):
    rows, raw = [], []
    import traceback as _tbr
    n_fail = 0
    for _, r in df.iterrows():
        try:
            ev = r.get('evidence', '')
            t = clean('TARGET', gen(prompt_for(r['comment'], ev, 'TARGET'), LIMITS['TARGET']))
            i = clean('INTENT', gen(prompt_for(r['comment'], ev, 'INTENT'), LIMITS['INTENT']))
            m = clean('IMPLICATION', gen(prompt_for(r['comment'], ev, 'IMPLICATION'), LIMITS['IMPLICATION']))
        except Exception as _er:
            n_fail += 1
            open(f'{OUT}/row_errors.jsonl', 'a').write(json.dumps(
                {'id': str(r.get('id', len(rows))), 'err': str(_er)[:500]}) + '\n')
            t, i, m = 'Unknown', 'To spread a misleading narrative.', 'Could erode trust.'
            ev = r.get('evidence', '')
        rows.append({'Index': r.get('Index', r.get('id', len(rows))), 'comment': r['comment'], 'target': t, 'intent': i, 'implication': m})
        raw.append({'Index': rows[-1]['Index'], 'comment': r['comment'], 'target': t, 'intent': i, 'implication': m})
    print(f'{split}: {n_fail} row fallbacks')
    pd.DataFrame(rows).to_csv(f'{OUT}/preds_{split}.csv', index=False)
    json.dump(raw, open(f'{OUT}/raw_outputs_{split}.jsonl','w'), indent=1)
    print(f'{split}: {len(rows)} -> preds_{split}.csv + raw_outputs_{split}.jsonl')
    open(f'{OUT}/stages.txt', 'a').write(f'c8-{split}-done\n')
    return pd.DataFrame(rows)
import traceback as _tb8
try:
    val_pred = run(val_df, 'val')
    test_pred = run(test_df if test_df is not None else val_df, 'test')
except Exception as _e8:
    open(f'{OUT}/c8_error.txt', 'w').write(str(_e8)[:1000] + '\n' + _tb8.format_exc(limit=10))
    print('C8 FAILED, see c8_error.txt:', str(_e8)[:300])
    raise
display(val_pred.head(3))

In [ ]:
open(f'{OUT}/stages.txt', 'a').write('c9-start\n')
# CELL 9 - eval: SBERT cosine + ROUGE-L F1 per field (Codabench replica) + plots
!pip install -q rouge-score sentence-transformers matplotlib 2>&1 | tail -1
import pandas as pd, numpy as np, json
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
ref = val_df; hyp = pd.read_csv(f'{OUT}/preds_val.csv')
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
res = {}
for f in ['target','intent','implication']:
    R, P = ref[f].astype(str).tolist(), hyp[f].astype(str).tolist()
    rl = [scorer.score(r,p)['rougeL'].fmeasure for r,p in zip(R,P)]
    E = sbert.encode(P, normalize_embeddings=True, show_progress_bar=False)
    G = sbert.encode(R, normalize_embeddings=True, show_progress_bar=False)
    sb = [float(np.dot(a,b)) for a,b in zip(E,G)]
    res[f] = {'sbert': float(np.mean(sb)), 'rougeL': float(np.mean(rl))}
    print(f"{f:12s} SBERT {res[f]['sbert']:.4f}  ROUGE-L {res[f]['rougeL']:.4f}")
res['mean'] = {'sbert': float(np.mean([res[k]['sbert'] for k in res])),
                 'rougeL': float(np.mean([res[k]['rougeL'] for k in res]))}
res['mean']['combined'] = (res['mean']['sbert']+res['mean']['rougeL'])/2
json.dump(res, open(f'{OUT}/metrics.json','w'), indent=2); print(json.dumps(res, indent=2))
from huggingface_hub import HfApi
try:
    HfApi().upload_file(path_or_fileobj=f"{OUT}/metrics.json", path_in_repo='metrics.json', repo_id=HF_REPO, token=os.environ['HF_TOKEN'])
    HfApi().upload_file(path_or_fileobj=f"{OUT}/preds_val.csv", path_in_repo='preds_val.csv', repo_id=HF_REPO, token=os.environ['HF_TOKEN'])
    print('metrics pushed to Hub')
except Exception as e: print(str(e)[:200])
import matplotlib.pyplot as plt
plt.figure(); plt.bar([f"{k}\nSBERT" for k in ['target','intent','implication']], [res[k]['sbert'] for k in ['target','intent','implication']]); plt.savefig(f'{OUT}/sbert.png'); plt.close()
plt.figure(); plt.bar([f"{k}\nR-L" for k in ['target','intent','implication']], [res[k]['rougeL'] for k in ['target','intent','implication']]); plt.savefig(f'{OUT}/rougeL.png'); plt.close()
print('plots saved')
open(f'{OUT}/stages.txt', 'a').write('c9-done\n')


In [ ]:
# CELL 10 - official-format submissions (xlsx Index/Comments/Target/Intent/Implication + splits)
open(f'{OUT}/stages.txt', 'a').write('c10-start\n')
import pandas as pd, zipfile, os
t = pd.read_csv(f'{OUT}/preds_test.csv')
test_ref = pd.read_csv(f'{OUT}/test_norm.csv')
t['Index'] = test_ref['Index'].tolist()[:len(t)]
t['Comments'] = test_ref['comment'].tolist()[:len(t)]
assert t[['Index','target','intent','implication']].notna().all().all(), 'empty cells!'
assert len(t) == len(test_ref), 'row-count mismatch vs test'
sub = pd.DataFrame({'Index': t['Index'], 'Comments': t['Comments'],
                    'Target': t['target'], 'Intent': t['intent'], 'Implication': t['implication']})
sub.to_excel(f'{OUT}/submission_official.xlsx', index=False)
sub[['Index','Target']].to_csv(f'{OUT}/submission_taskA.csv', index=False)
sub[['Index','Intent','Implication']].to_csv(f'{OUT}/submission_taskB.csv', index=False)
for name in ['submission_taskA.csv','submission_taskB.csv']:
    with zipfile.ZipFile(f"{OUT}/{name.replace('.csv','.zip')}", 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(f'{OUT}/{name}', arcname=name)
    print(name, os.path.getsize(f"{OUT}/{name.replace('.csv','.zip')}"), 'bytes')
print('official xlsx rows:', len(sub), '| upload submission_official.xlsx / zips to Codabench')
from huggingface_hub import HfApi
for f in ['submission_official.xlsx','submission_taskA.zip','submission_taskB.zip','raw_outputs_test.jsonl']:
    try: HfApi().upload_file(path_or_fileobj=f'{OUT}/{f}', path_in_repo=f, repo_id=HF_REPO, token=os.environ['HF_TOKEN'])
    except Exception as e: print(f, str(e)[:150])
print('Hub backup done. Elapsed hrs:', round((__import__("time").time()-T0)/3600, 2))
open(f'{OUT}/stages.txt', 'a').write('c10-done\n')


In [ ]:
# CELL 11 - resume playbook (run this first in the NEXT session)
print('NEXT SESSION (2 min): set GPU T4 x2, add HF_TOKEN secret, run cells 1-3, then:')
print(f"from huggingface_hub import snapshot_download; snapshot_download('{HF_REPO}', local_dir='{OUT}/resume', token=os.environ['HF_TOKEN'])")
print('Cell 6 auto-resumes via resume_ckpt. If converged, skip to cells 8-10 for eval-only.')
print('Ablations: WITH_RAG False | MODEL_ID=mistralai/Mistral-7B-Instruct-v0.3 (Implication expert) | ensemble in cell 8.')